In [1]:
import os, glob
import pandas as pd

In [2]:
train_files = glob.glob("D:/Coding/Project/train/*.csv")
test_files = glob.glob("D:/Coding/Project/test/*.csv")

In [3]:
print(len(train_files),len(test_files))
print(test_files[:5])

51 21
['D:/Coding/Project/test\\ARP_Spoofing_test.pcap.csv', 'D:/Coding/Project/test\\Benign_test.pcap.csv', 'D:/Coding/Project/test\\MQTT-DDoS-Connect_Flood_test.pcap.csv', 'D:/Coding/Project/test\\MQTT-DDoS-Publish_Flood_test.pcap.csv', 'D:/Coding/Project/test\\MQTT-DoS-Connect_Flood_test.pcap.csv']


In [4]:
sizes = [os.path.getsize(f)/1e6 for f in train_files]
print(min(sizes), max(sizes), sum(sizes))

0.210439 60.629854 1864.515482


In [5]:
df_sample = pd.read_csv(train_files[0])
print(df_sample.shape)
print(df_sample.columns.tolist())
df_sample.head()

(16047, 45)
['Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight']


,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,...,AVG,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight
0,866.6,10.4,64.0,45722.390222,45722.390222,0.0,0.0,0.0,0.0,0.3,...,161.829921,150.681076,431.8,1.694026e+08,5.5,16.963645,213.095221,66236.076476,0.9,38.5
1,3934.3,12.6,131.2,35708.799475,35708.799475,0.0,0.0,0.0,0.0,0.3,...,475.706207,500.702909,406.3,1.694026e+08,13.5,30.885371,708.919620,251721.126817,1.0,244.6
2,5592.8,12.6,97.6,66.403506,66.403506,0.0,0.0,0.0,0.0,0.3,...,249.303651,205.552035,386.6,1.361110e-02,5.5,21.787095,290.694475,84028.647525,0.9,38.5
3,9303.6,14.8,80.8,51.201280,51.201280,0.0,0.0,0.0,0.0,0.1,...,361.952562,421.683660,300.2,1.694026e+08,13.5,26.954506,597.046005,178453.001691,1.0,244.6
4,8592.4,12.6,98.6,42.706455,42.706455,0.0,0.0,0.0,0.0,0.2,...,314.474921,265.394239,209.2,1.393099e-02,5.5,24.255148,375.324132,80115.110731,0.9,38.5


In [6]:
for f in train_files[:15]:
    print(os.path.basename(f))

ARP_Spoofing_train.pcap.csv
Benign_train.pcap.csv
MQTT-DDoS-Connect_Flood_train.pcap.csv
MQTT-DDoS-Publish_Flood_train.pcap.csv
MQTT-DoS-Connect_Flood_train.pcap.csv
MQTT-DoS-Publish_Flood_train.pcap.csv
MQTT-Malformed_Data_train.pcap.csv
Recon-OS_Scan_train.pcap.csv
Recon-Ping_Sweep_train.pcap.csv
Recon-Port_Scan_train.pcap.csv
Recon-VulScan_train.pcap.csv
TCP_IP-DDoS-ICMP1_train.pcap.csv
TCP_IP-DDoS-ICMP2_train.pcap.csv
TCP_IP-DDoS-ICMP3_train.pcap.csv
TCP_IP-DDoS-ICMP4_train.pcap.csv


In [7]:
def load_split(files, limit=None):
    dfs = []
    for f in (files[:limit] if limit else files):
        d = pd.read_csv(f)
        d["label"] = os.path.basename(f).split("_train")[0].split("_test")[0]
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

train_sample = load_split(train_files, limit=10)
train_sample["label"].value_counts()

label
Benign                     192732
MQTT-DDoS-Connect_Flood    173036
Recon-Port_Scan             83981
MQTT-DoS-Publish_Flood      44376
MQTT-DDoS-Publish_Flood     27623
Recon-OS_Scan               16832
ARP_Spoofing                16047
MQTT-DoS-Connect_Flood      12773
MQTT-Malformed_Data          5130
Recon-Ping_Sweep              740
Name: count, dtype: int64

In [8]:
train_df = load_split(train_files)
test_df = load_split(test_files)

train_df.to_parquet("train.parquet")
test_df.to_parquet("test.parquet")

In [9]:
import duckdb
duckdb.sql("SELECT label, COUNT(*) FROM train_df GROUP BY label ORDER BY 2 DESC").show()

┌─────────────────────────┬──────────────┐
│          label          │ count_star() │
│         varchar         │    int64     │
├─────────────────────────┼──────────────┤
│ TCP_IP-DDoS-UDP2        │       207295 │
│ TCP_IP-DDoS-UDP3        │       206604 │
│ TCP_IP-DDoS-UDP4        │       206343 │
│ TCP_IP-DDoS-UDP1        │       206170 │
│ TCP_IP-DDoS-UDP5        │       205507 │
│ TCP_IP-DDoS-UDP8        │       204105 │
│ TCP_IP-DDoS-TCP3        │       204075 │
│ TCP_IP-DDoS-SYN2        │       203669 │
│ TCP_IP-DDoS-TCP1        │       202311 │
│ TCP_IP-DDoS-UDP6        │       202247 │
│        ·                │          ·   │
│        ·                │          ·   │
│        ·                │          ·   │
│ TCP_IP-DoS-TCP4         │        93488 │
│ Recon-Port_Scan         │        83981 │
│ MQTT-DoS-Publish_Flood  │        44376 │
│ MQTT-DDoS-Publish_Flood │        27623 │
│ Recon-OS_Scan           │        16832 │
│ ARP_Spoofing            │        16047 │
│ MQTT-DoS-

In [ ]:
def del_last_digit(list):
    